In [ ]:

#--------------------------------------------------------------------
# Exercise 2.1.
# Wine Quality Data Set: "data/wines.csv" source: https://archive.ics.uci.edu/ml/datasets/wine+quality
# The file contains data on samples of white and red Portuguese wine Vinho Verde.
# Various physico-chemical characteristics of individual samples are available as well as wine quality scores on a point scale (0-10) made by specialists.
# estimate the linear regression model with the quality evaluation as the dependent variable, treating the explained variable as quantitative.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

wines = pd.read_csv("wines.csv")

In [2]:
pd.set_option("display.max_columns",50)
wines.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,type,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,red,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,red,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,red,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,red,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,red,5


In [3]:
import statsmodels.api as sm
mod = sm.OLS(wines["quality"], wines[["fixed_acidity",  "volatile_acidity", "citric_acid", "residual_sugar",
                  "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide", "density",
                  "pH", "sulphates", "alcohol"]])
res = mod.fit()
res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                                 OLS Regression Results                                
=======================================================================================
Dep. Variable:                quality   R-squared (uncentered):                   0.984
Model:                            OLS   Adj. R-squared (uncentered):              0.984
Method:                 Least Squares   F-statistic:                          3.710e+04
Date:                Sat, 17 Jan 2026   Prob (F-statistic):                        0.00
Time:                        07:23:51   Log-Likelihood:                         -7226.4
No. Observations:                6497   AIC:                                  1.447e+04
Df Residuals:                    6486   BIC:                                  1.455e+04
Df Model:                          11                                                  
Covariance Type:            nonrobust                                                  
========================================================================================
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
fixed_acidity            0.0100      0.010      1.047      0.295      -0.009       0.029
volatile_acidity        -1.4573      0.072    -20.130      0.000      -1.599      -1.315
citric_acid             -0.1137      0.080     -1.426      0.154      -0.270       0.043
residual_sugar           0.0221      0.002      9.259      0.000       0.017       0.027
chlorides               -0.7955      0.326     -2.436      0.015      -1.436      -0.155
free_sulfur_dioxide      0.0060      0.001      7.966      0.000       0.005       0.007
total_sulfur_dioxide    -0.0022      0.000     -8.228      0.000      -0.003      -0.002
density                  1.9225      0.281      6.837      0.000       1.371       2.474
pH                       0.1641      0.069      2.384      0.017       0.029       0.299
sulphates                0.6408      0.071      8.998      0.000       0.501       0.780
alcohol                  0.3333      0.009     37.212      0.000       0.316       0.351
==============================================================================
Omnibus:                      140.142   Durbin-Watson:                   1.647
Prob(Omnibus):                  0.000   Jarque-Bera (JB):              311.565
Skew:                           0.010   Prob(JB):                     2.21e-68
Kurtosis:                       4.073   Cond. No.                     4.98e+03
==============================================================================

Notes:
[1] R² is computed without centering (uncentered) since the model does not contain a constant.
[2] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[3] The condition number is large, 4.98e+03. This might indicate that there are
strong multicollinearity or other numerical problems.
"""

In [18]:
import numpy as np
import pandas as pd

from sklearn.model_selection import (
    KFold,
    RepeatedKFold,
    StratifiedKFold,
    RepeatedStratifiedKFold,
    cross_val_score,
    train_test_split
)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder


In [21]:
X = wines.drop(columns=["quality"])
y = wines["quality"]
num_cols = [
    "fixed_acidity", "volatile_acidity", "citric_acid", "residual_sugar",
    "chlorides", "free_sulfur_dioxide", "total_sulfur_dioxide", "density",
    "pH", "sulphates", "alcohol"
]

cat_cols = ["type"]

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(drop="first"), cat_cols)
    ]
)

In [23]:
models = {
    "Logistic Regression": Pipeline([
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=1000))
    ]),

    "SVM": Pipeline([
        ("prep", preprocessor),
        ("model", SVC(C=1, kernel="rbf"))
    ]),

    "Random Forest": Pipeline([
        ("prep", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            random_state=42
        ))
    ])
}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

print(" 5-Fold CV ")
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=kf, scoring="accuracy")
    print(f"{name}: Mean={scores.mean():.4f}, Std={scores.std():.4f}")

rkf = RepeatedKFold(
    n_splits=5,
    n_repeats=10,
    random_state=42
)

print("\n Repeated 5-Fold CV ")
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=rkf, scoring="accuracy")
    print(f"{name}: Mean={scores.mean():.4f}, Std={scores.std():.4f}")

 5-Fold CV 
Logistic Regression: Mean=0.5441, Std=0.0089
SVM: Mean=0.5697, Std=0.0151
Random Forest: Mean=0.5698, Std=0.0097

 Repeated 5-Fold CV 
Logistic Regression: Mean=0.5427, Std=0.0093
SVM: Mean=0.5712, Std=0.0120
Random Forest: Mean=0.5691, Std=0.0109


In [25]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

rf_pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        random_state=42
    ))
])

rf_pipeline.fit(X_train, y_train)

train_acc = accuracy_score(y_train, rf_pipeline.predict(X_train))
test_acc = accuracy_score(y_test, rf_pipeline.predict(X_test))

print(" Overfitting Check")
print(f"Train accuracy: {train_acc:.4f}")
print(f"Test accuracy : {test_acc:.4f}")

 Overfitting Check
Train accuracy: 0.5805
Test accuracy : 0.5431


In [ ]:
#Exercise 2.2.
#Titanic passengers data – 1310 observations and 15 variables:

#passenger_id – Unique passenger id
#pclass – Ticket class (1 = 1st, 2 = 2nd, 3 = 3rd)
#survived – Survival (0 = No, 1 = Yes)
#name – Name and SUrname
#sex – Sex (0 = Male, 1 = Female)
#age – Age in years
#sibsp – of siblings / spouses aboard the Titanic
#parch – of parents / children aboard the Titanic
#ticket – Ticket number
#fare – Passenger fare
#cabin – Cabin number
#embarked – Port of Embarkation (C = Cherbourg, Q = Queenstown, S = Southampton)
#boat – Lifeboat (if survived)
#body – Body number (if did not survive and body was recovered)
#home.dest – Home/Destination
#estimate logistic regression model that can be used to explain the probability of survival (survived = 1).

#lets load medical_care data after transformations applied last week

In [7]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

titanic = pd.read_csv("titanic.csv")

In [8]:
pd.set_option("display.max_columns",50)
titanic.head()

,passenger_id,pclass,survived,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,boat,body,home.dest
0,1,1,1,"Allen, Miss. Elisabeth Walton",1,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,2,1,1,"Allison, Master. Hudson Trevor",0,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,3,1,0,"Allison, Miss. Helen Loraine",1,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,4,1,0,"Allison, Mr. Hudson Joshua Creighton",0,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,5,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",1,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


In [9]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

mod = smf.glm(formula="survived ~ pclass + sex + age + sibsp  + embarked", data=titanic, family=sm.families.Binomial())
res = mod.fit()
res.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                 Generalized Linear Model Regression Results                  
==============================================================================
Dep. Variable:               survived   No. Observations:                 1044
Model:                            GLM   Df Residuals:                     1037
Model Family:                Binomial   Df Model:                            6
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -477.59
Date:                Sat, 17 Jan 2026   Deviance:                       955.18
Time:                        07:35:24   Pearson chi2:                 1.07e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.3538
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
Intercept         2.6730      0.407      6.573      0.000       1.876       3.470
embarked[T.Q]    -1.4720      0.444     -3.315      0.001      -2.342      -0.602
embarked[T.S]    -0.6921      0.207     -3.345      0.001      -1.098      -0.287
pclass           -1.0240      0.117     -8.756      0.000      -1.253      -0.795
sex               2.6300      0.176     14.925      0.000       2.285       2.975
age              -0.0379      0.007     -5.721      0.000      -0.051      -0.025
sibsp            -0.3271      0.102     -3.193      0.001      -0.528      -0.126
=================================================================================
"""

In [17]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
import random
from sklearn.linear_model import LinearRegression

df = titanic.sample(frac=1, random_state=42).reset_index(drop=True)

cols_to_drop = ['passenger_id', 'name', 'ticket', 'cabin', 'boat', 'home.dest', 'body']
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

df.fillna(0, inplace=True)


df = pd.get_dummies(df, drop_first=True)


df.columns = [col.replace('.', '_').replace(' ', '_') for col in df.columns]

target_col = 'survived'

train, test = train_test_split(df,
                               test_size=0.3,
                               random_state=random.randint(0, 1000))

print(f"Train shape: {train.shape}, Test shape: {test.shape}")

k_fold = 5
fold_size = len(df) // k_fold

X = df.drop(columns=['survived'])
y = df['survived']

for k in range(k_fold):
    start = k * fold_size
    end = (k + 1) * fold_size

    X_test = X.iloc[start:end]
    y_test = y.iloc[start:end]

    X_train = pd.concat([X.iloc[:start], X.iloc[end:]])
    y_train = pd.concat([y.iloc[:start], y.iloc[end:]])

    model=LinearRegression()
    model.fit(X_train, y_train)
    y_pred=model.predict(X_test)
    r2=r2_score(y_test, y_pred)
    print(f"Fold {k + 1}: R-squared = {r2} ,  RMSE = {np.sqrt(mean_squared_error(y_test, y_pred))}")

Train shape: (916, 10), Test shape: (393, 10)
Fold 1: R-squared = 0.3324730271550226 ,  RMSE = 0.40631988703067135
Fold 2: R-squared = 0.37239043243912096 ,  RMSE = 0.38283537559467823
Fold 3: R-squared = 0.35109984465022537 ,  RMSE = 0.3838783227238632
Fold 4: R-squared = 0.4218041969680031 ,  RMSE = 0.35845216728223683
Fold 5: R-squared = 0.23925991163743054 ,  RMSE = 0.4289729359467673


In [ ]:

# Exercise 2.3.
# Wine Quality Data Set: "data/wines.csv"

# estimate the multinomial logistic regression model with the quality evaluation as the dependent variable,# treating the explained variable as qualitative.

In [ ]:
import patsy

# Create the design matrix for exogenous variables using patsy.dmatrix
exog_formula = "fixed_acidity + volatile_acidity + citric_acid + residual_sugar + chlorides + free_sulfur_dioxide + total_sulfur_dioxide + density + pH + sulphates + alcohol"
exog = patsy.dmatrix(exog_formula, data=wines, return_type='dataframe')

# Create the endog variable (dependent variable)
endog = wines['quality']

# Now create the MNLogit model by passing endog and exog separately
mod = sm.MNLogit(endog, exog)
res = mod.fit()
res.summary()

         Current function value: 1.064719
         Iterations: 35


/usr/local/lib/python3.12/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


<class 'statsmodels.iolib.summary.Summary'>
"""
                          MNLogit Regression Results                          
==============================================================================
Dep. Variable:                quality   No. Observations:                 6497
Model:                        MNLogit   Df Residuals:                     6425
Method:                           MLE   Df Model:                           66
Date:                Mon, 12 Jan 2026   Pseudo R-squ.:                  0.1641
Time:                        02:12:39   Log-Likelihood:                -6917.5
converged:                      False   LL-Null:                       -8275.4
Covariance Type:            nonrobust   LLR p-value:                     0.000
========================================================================================
           quality=4       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             -203.7363    225.259     -0.904      0.366    -645.236     237.764
fixed_acidity           -0.9280      0.282     -3.289      0.001      -1.481      -0.375
volatile_acidity        -1.0093      1.049     -0.962      0.336      -3.065       1.046
citric_acid              1.1892      1.699      0.700      0.484      -2.141       4.520
residual_sugar          -0.1367      0.098     -1.392      0.164      -0.329       0.056
chlorides              -15.3111      4.884     -3.135      0.002     -24.884      -5.738
free_sulfur_dioxide     -0.0817      0.012     -6.620      0.000      -0.106      -0.058
total_sulfur_dioxide     0.0036      0.006      0.630      0.529      -0.008       0.015
density                231.3114    229.263      1.009      0.313    -218.035     680.658
pH                      -4.4241      1.884     -2.349      0.019      -8.116      -0.732
sulphates                2.8116      2.127      1.322      0.186      -1.356       6.980
alcohol                 -0.0943      0.356     -0.265      0.791      -0.793       0.604
----------------------------------------------------------------------------------------
           quality=5       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             -535.5121    206.158     -2.598      0.009    -939.574    -131.451
fixed_acidity           -1.1766      0.256     -4.600      0.000      -1.678      -0.675
volatile_acidity        -3.2259      0.991     -3.254      0.001      -5.169      -1.283
citric_acid              1.1076      1.610      0.688      0.491      -2.047       4.262
residual_sugar          -0.2296      0.090     -2.537      0.011      -0.407      -0.052
chlorides              -12.2898      3.989     -3.081      0.002     -20.108      -4.472
free_sulfur_dioxide     -0.0328      0.010     -3.246      0.001      -0.053      -0.013
total_sulfur_dioxide    -0.0013      0.005     -0.240      0.811      -0.012       0.009
density                571.7081    209.716      2.726      0.006     160.673     982.743
pH                      -5.6482      1.750     -3.228      0.001      -9.078      -2.219
sulphates                3.4029      2.012      1.692      0.091      -0.540       7.346
alcohol                  0.0326      0.331      0.098      0.922      -0.616       0.681
----------------------------------------------------------------------------------------
           quality=6       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
Intercept             -509.1501    206.119     -2.470      0.014    -913.135    -105.165
fixed_acidity           -1.1359      0.255     -4.454      0.000      -1.636      -0.636
volatile_acidity        -7.0468      1.012     -6.963      0.000      -9.030      -5.063
citric_acid              0.5296      